In [1]:
from google.colab import drive
import json
import numpy as np
import pandas as pd
import torch

!pip -q install -U transformers datasets accelerate evaluate scikit-learn
!pip install -q wandb
!pip install -q pysentimiento
!pip install -q "pyarrow>=14.0.0" --upgrade

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
drive.mount('/content/drive')

ruta = "/content/drive/MyDrive/Tesis ORT Benedetti La Laina/df_final_para_modelo.jsonl"

df_limpio = pd.read_json(ruta, lines=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import wandb

os.environ["WANDB_API_KEY"] = "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"
wandb.login()  # la toma desde el env


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: flalaina (fbenedetti97-universidad-ort-uruguay) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
from sklearn.metrics import classification_report, f1_score, accuracy_score
import numpy as np
import wandb

def one_vs_rest_accuracy(y_true, y_pred, class_id):
    y_true_bin = (np.array(y_true) == class_id).astype(int)
    y_pred_bin = (np.array(y_pred) == class_id).astype(int)
    return accuracy_score(y_true_bin, y_pred_bin)

def build_metrics_dict(y_true, y_pred, label_list, label2id, split, variant):
    """
    split: 'eval' o 'test'
    variant: 'simple' o 'sliding'
    """
    report = classification_report(
        y_true,
        y_pred,
        labels=[label2id[lbl] for lbl in label_list],
        target_names=label_list,
        output_dict=True,
        zero_division=0
    )

    metrics = {
        f"{split}/accuracy_{variant}": accuracy_score(y_true, y_pred),
        f"{split}/f1_macro_{variant}": f1_score(y_true, y_pred, average="macro"),
        f"{split}/f1_weighted_{variant}": f1_score(y_true, y_pred, average="weighted"),
        f"{split}/confusion_matrix_{variant}": wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=label_list
        ),
    }

    for lbl in label_list:
        metrics[f"{split}/class_{lbl}_precision_{variant}"] = float(report[lbl]["precision"])
        metrics[f"{split}/class_{lbl}_recall_{variant}"] = float(report[lbl]["recall"])
        metrics[f"{split}/class_{lbl}_f1_{variant}"] = float(report[lbl]["f1-score"])
        metrics[f"{split}/class_{lbl}_support_{variant}"] = int(report[lbl]["support"])
        metrics[f"{split}/class_{lbl}_accuracy_ovr_{variant}"] = float(
            one_vs_rest_accuracy(y_true, y_pred, label2id[lbl])
        )

    return metrics, report

# ROBERTUITO BASELINE SIN ENTRENAR CON INTERVENCIONES PARLAMENTARIAS


In [ ]:
# =========================================
# 0) W&B + Config
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

WANDB_PROJECT = "Tesis ORT"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

if wandb.run is not None:
    wandb.finish()

SEED = 1899
RUN_NAME = f"robertuito_sentiment_ckpt_{SEED}"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

run = wandb.init(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": "pysentimiento/robertuito-sentiment-analysis",
        "run_type": "pretrained_checkpoint_inference_only",
        "uses_training_data": False,
        "uses_sliding": False,
        "selection_metric": "test/f1_macro_simple",
    },
    tags=["baseline", "inference_only", "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id)

# =========================================
# 1) Instalar / imports del modelo
# =========================================
# Si estás en Colab y no lo tenés instalado:
# !pip install -q pysentimiento

from pysentimiento import create_analyzer

# NOTA:
# El analyzer trunca internamente a 128 tokens.
# Para textos largos (intervenciones parlamentarias) esto puede afectar la calidad.
# No hay sliding window disponible con esta API: es una limitación del baseline.
analyzer = create_analyzer(task="sentiment", lang="es")

# =========================================
# 2) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Mismo split 80/10/10 que el resto de los experimentos
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 3) Inferencia
# =========================================
# El checkpoint devuelve POS / NEG / NEU → mapeamos a tus etiquetas N / Neu / P
pred_map = {
    "NEG": "N",
    "NEU": "Neu",
    "POS": "P",
}

def predict_one(text: str) -> dict:
    out = analyzer.predict(text)

    raw_label = out.output
    probs_raw = out.probas   # {"POS": float, "NEG": float, "NEU": float}
    pred_label = pred_map[raw_label]

    p_neg = float(probs_raw.get("NEG", 0.0))
    p_neu = float(probs_raw.get("NEU", 0.0))
    p_pos = float(probs_raw.get("POS", 0.0))

    conf_pred = {
        "N": p_neg,
        "Neu": p_neu,
        "P": p_pos
    }[pred_label]

    return {
        "pred_label": pred_label,
        "conf_pred": float(conf_pred),
        "proba_N": p_neg,
        "proba_Neu": p_neu,
        "proba_P": p_pos,
        "proba_vector": [p_neg, p_neu, p_pos],
        "raw_label": raw_label,
    }

# =========================================
# 4) VALIDATION — solo referencia, NO selección final
# =========================================
print("\nInferencia sobre validation...")
val_meta = val_df.reset_index(drop=True).copy()
val_preds = [predict_one(txt) for txt in val_meta["text"].tolist()]
val_meta = pd.concat([val_meta, pd.DataFrame(val_preds)], axis=1)
val_meta["true_label"] = val_meta["sentimiento"]

y_true_val = val_meta["true_label"].map(label2id).to_numpy()
y_pred_val = val_meta["pred_label"].map(label2id).to_numpy()

val_metrics, val_report = build_metrics_dict(
    y_true=y_true_val,
    y_pred=y_pred_val,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(val_metrics)

print(f"  Val accuracy: {val_metrics['eval/accuracy_simple']:.4f} | "
      f"F1-macro: {val_metrics['eval/f1_macro_simple']:.4f}")

print("\nValidation classification report:")
print(classification_report(
    y_true_val,
    y_pred_val,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 5) TEST — número final que va a la tabla
# =========================================
print("\nInferencia sobre test...")
test_meta = test_df.reset_index(drop=True).copy()
test_preds = [predict_one(txt) for txt in test_meta["text"].tolist()]
test_meta = pd.concat([test_meta, pd.DataFrame(test_preds)], axis=1)
test_meta["true_label"] = test_meta["sentimiento"]

y_true = test_meta["true_label"].map(label2id).to_numpy()
y_pred = test_meta["pred_label"].map(label2id).to_numpy()

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()

# =========================================
# 6) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label", "pred_label", "conf_pred",
    "proba_N", "proba_Neu", "proba_P", "proba_vector", "raw_label",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")

print("Excel generado:", out_path)

# =========================================
# 7) Log W&B + resumen final
# =========================================
test_metrics, test_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_metrics)

print("\n" + classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

print("\n" + "=" * 50)
print("RESULTADOS FINALES — TEST SET")
print(f"  Accuracy:    {test_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro:    {test_metrics['test/f1_macro_simple']:.4f}")
print(f"  F1-weighted: {test_metrics['test/f1_weighted_simple']:.4f}")
print(f"  Errores:     {len(df_errors_simple)} / {len(test_meta)}")
print("=" * 50)

for lbl in label_list:
    print(
        f"[TEST] {lbl} | "
        f"Precision: {test_metrics[f'test/class_{lbl}_precision_simple']:.4f} | "
        f"Recall: {test_metrics[f'test/class_{lbl}_recall_simple']:.4f} | "
        f"F1: {test_metrics[f'test/class_{lbl}_f1_simple']:.4f} | "
        f"Support: {int(test_metrics[f'test/class_{lbl}_support_simple'])} | "
        f"Acc OvR: {test_metrics[f'test/class_{lbl}_accuracy_ovr_simple']:.4f}"
    )

wandb.finish()

RUN: robertuito_sentiment_ckpt_1899 ID: py9566uq


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

Train: 979 Val: 122 Test: 123

Inferencia sobre validation...
  Val accuracy: 0.6557 | F1-macro: 0.6421

Validation classification report:
              precision    recall  f1-score   support

           N       0.64      0.71      0.67        41
         Neu       0.60      0.80      0.69        45
           P       0.88      0.42      0.57        36

    accuracy                           0.66       122
   macro avg       0.71      0.64      0.64       122
weighted avg       0.70      0.66      0.65       122


Inferencia sobre test...
Excel generado: analisis_robertuito_sentiment_ckpt_1899.xlsx

              precision    recall  f1-score   support

           N       0.68      0.63      0.66        41
         Neu       0.49      0.80      0.61        46
           P       1.00      0.28      0.43        36

    accuracy                           0.59       123
   macro avg       0.73      0.57      0.57       123
weighted avg       0.71      0.59      0.58       123


RESULTADOS

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy_simple,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_f1_simple,▁
eval/class_N_precision_simple,▁
eval/class_N_recall_simple,▁
eval/class_N_support_simple,▁
eval/class_Neu_accuracy_ovr_simple,▁
+29,...


# Robertuito SEED 1899

## Truncado

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = False
AGG_METHOD  = "vote"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1899
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
RUN_NAME   = (
    f"robertuito_base_{SEED}"
    if not USE_SLIDING
    else f"robertuito_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": MODEL_CKPT,
        "run_type": "finetuned_single_run",
        "max_len_train": MAX_LEN_TRAIN,
        "use_sliding": USE_SLIDING,
        "max_len_sw": MAX_LEN_SW,
        "stride_sw": STRIDE_SW,
        "agg_method": AGG_METHOD,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 4,
        "weight_decay": 0.01,
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "sliding" if USE_SLIDING else "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(
    text,
    model,
    tokenizer,
    id2label,
    max_len=128,
    stride=64,
    agg="mean"
):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"robertuito_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Usar siempre el mejor checkpoint cargado por Trainer
best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("\nExcel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

# =========================================
# 14) Resumen final legible
# =========================================
print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: robertuito_base_1899 ID: qzhy37z8 USE_SLIDING: False AGG: vote
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3282', 'grad_norm': '11.02', 'learning_rate': '3.952e-06', 'epoch': '3.226'}
{'eval_loss': '0.6831', 'eval_accuracy': '0.7541', 'eval_f1_macro': '0.7509', 'eval_f1_weighted': '0.7557', 'eval_runtime': '0.3922', 'eval_samples_per_second': '311.1', 'eval_steps_per_second': '10.2', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '135.4', 'train_samples_per_second': '28.92', 'train_steps_per_second': '1.831', 'train_loss': '0.4631', 'epoch': '4'}

VALIDATION SIMPLE
  Accuracy: 0.7869
  F1-macro: 0.7837
              precision    recall  f1-score   support

           N       0.73      0.80      0.77        41
         Neu       0.86      0.82      0.84        45
           P       0.76      0.72      0.74        36

    accuracy                           0.79       122
   macro avg       0.79      0.78      0.78       122
weighted avg       0.79      0.79      0.79       122


TEST SIMPLE
  Accuracy: 0.7154
  F1-macro: 0.7102
              precision    recall  f1-score   support

           N       0.66      0.76      0.70        41
         Neu       0.78      0.76      0.77        46
           P       0.71      0.61      0.66        36

    accuracy                           0.72       123
   macro avg       0.72      0.71      0.71       123
weighted avg       0.72      0.72      0.71     

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,██▁▁
eval/accuracy_simple,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_f1_simple,▁
eval/class_N_precision_simple,▁
eval/class_N_recall_simple,▁
eval/class_N_support_simple,▁
+48,...


## Slide + Vote

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "vote"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1899
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
RUN_NAME   = (
    f"robertuito_base_{SEED}"
    if not USE_SLIDING
    else f"robertuito_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": MODEL_CKPT,
        "run_type": "finetuned_single_run",
        "max_len_train": MAX_LEN_TRAIN,
        "use_sliding": USE_SLIDING,
        "max_len_sw": MAX_LEN_SW,
        "stride_sw": STRIDE_SW,
        "agg_method": AGG_METHOD,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 4,
        "weight_decay": 0.01,
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "sliding" if USE_SLIDING else "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(
    text,
    model,
    tokenizer,
    id2label,
    max_len=128,
    stride=64,
    agg="mean"
):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"robertuito_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Usar siempre el mejor checkpoint cargado por Trainer
best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("\nExcel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

# =========================================
# 14) Resumen final legible
# =========================================
print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

## Slide + Mean


In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1899
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
RUN_NAME   = (
    f"robertuito_base_{SEED}"
    if not USE_SLIDING
    else f"robertuito_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": MODEL_CKPT,
        "run_type": "finetuned_single_run",
        "max_len_train": MAX_LEN_TRAIN,
        "use_sliding": USE_SLIDING,
        "max_len_sw": MAX_LEN_SW,
        "stride_sw": STRIDE_SW,
        "agg_method": AGG_METHOD,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 4,
        "weight_decay": 0.01,
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "sliding" if USE_SLIDING else "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(
    text,
    model,
    tokenizer,
    id2label,
    max_len=128,
    stride=64,
    agg="mean"
):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"robertuito_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Usar siempre el mejor checkpoint cargado por Trainer
best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("\nExcel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

# =========================================
# 14) Resumen final legible
# =========================================
print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

# Robertuito SEED 1405


## Truncado

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = False
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1405
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
RUN_NAME   = (
    f"robertuito_base_{SEED}"
    if not USE_SLIDING
    else f"robertuito_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": MODEL_CKPT,
        "run_type": "finetuned_single_run",
        "max_len_train": MAX_LEN_TRAIN,
        "use_sliding": USE_SLIDING,
        "max_len_sw": MAX_LEN_SW,
        "stride_sw": STRIDE_SW,
        "agg_method": AGG_METHOD,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 4,
        "weight_decay": 0.01,
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "sliding" if USE_SLIDING else "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(
    text,
    model,
    tokenizer,
    id2label,
    max_len=128,
    stride=64,
    agg="mean"
):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"robertuito_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Usar siempre el mejor checkpoint cargado por Trainer
best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("\nExcel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

# =========================================
# 14) Resumen final legible
# =========================================
print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: robertuito_base_1405 ID: 8cgzl2m3 USE_SLIDING: False AGG: mean
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3245', 'grad_norm': '16.33', 'learning_rate': '3.952e-06', 'epoch': '3.226'}
{'eval_loss': '0.5971', 'eval_accuracy': '0.8115', 'eval_f1_macro': '0.8086', 'eval_f1_weighted': '0.811', 'eval_runtime': '0.2368', 'eval_samples_per_second': '515.3', 'eval_steps_per_second': '16.89', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '107.7', 'train_samples_per_second': '36.35', 'train_steps_per_second': '2.302', 'train_loss': '0.4549', 'epoch': '4'}

VALIDATION SIMPLE
  Accuracy: 0.8443
  F1-macro: 0.8431
              precision    recall  f1-score   support

           N       0.86      0.78      0.82        41
         Neu       0.87      0.87      0.87        45
           P       0.80      0.89      0.84        36

    accuracy                           0.84       122
   macro avg       0.84      0.85      0.84       122
weighted avg       0.85      0.84      0.84       122


TEST SIMPLE
  Accuracy: 0.6992
  F1-macro: 0.6940
              precision    recall  f1-score   support

           N       0.71      0.61      0.66        41
         Neu       0.80      0.78      0.79        46
           P       0.58      0.69      0.63        36

    accuracy                           0.70       123
   macro avg       0.70      0.70      0.69       123
weighted avg       0.71      0.70      0.70     

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▃█▁▁
eval/accuracy_simple,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_f1_simple,▁
eval/class_N_precision_simple,▁
eval/class_N_recall_simple,▁
eval/class_N_support_simple,▁
+48,...


## Slide + Vote

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "vote"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1405
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
RUN_NAME   = (
    f"robertuito_base_{SEED}"
    if not USE_SLIDING
    else f"robertuito_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": MODEL_CKPT,
        "run_type": "finetuned_single_run",
        "max_len_train": MAX_LEN_TRAIN,
        "use_sliding": USE_SLIDING,
        "max_len_sw": MAX_LEN_SW,
        "stride_sw": STRIDE_SW,
        "agg_method": AGG_METHOD,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 4,
        "weight_decay": 0.01,
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "sliding" if USE_SLIDING else "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(
    text,
    model,
    tokenizer,
    id2label,
    max_len=128,
    stride=64,
    agg="mean"
):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"robertuito_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Usar siempre el mejor checkpoint cargado por Trainer
best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("\nExcel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

# =========================================
# 14) Resumen final legible
# =========================================
print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: robertuito_slide_1405_vote ID: rb2cjrfk USE_SLIDING: True AGG: vote
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3245', 'grad_norm': '16.33', 'learning_rate': '3.952e-06', 'epoch': '3.226'}
{'eval_loss': '0.5971', 'eval_accuracy': '0.8115', 'eval_f1_macro': '0.8086', 'eval_f1_weighted': '0.811', 'eval_runtime': '0.247', 'eval_samples_per_second': '494', 'eval_steps_per_second': '16.2', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '99.77', 'train_samples_per_second': '39.25', 'train_steps_per_second': '2.486', 'train_loss': '0.4549', 'epoch': '4'}

VALIDATION SIMPLE
  Accuracy: 0.8443
  F1-macro: 0.8431
              precision    recall  f1-score   support

           N       0.86      0.78      0.82        41
         Neu       0.87      0.87      0.87        45
           P       0.80      0.89      0.84        36

    accuracy                           0.84       122
   macro avg       0.84      0.85      0.84       122
weighted avg       0.85      0.84      0.84       122


VALIDATION SLIDING
  Accuracy: 0.8197
  F1-macro: 0.8158
              precision    recall  f1-score   support

           N       0.75      0.88      0.81        41
         Neu       0.90      0.84      0.87        45
           P       0.81      0.72      0.76        36

    accuracy                           0.82       122
   macro avg       0.82      0.81      0.82       122
weighted avg       0.83      0.82      0.

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▃█▁▁
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


## Slide + Mean

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 1405
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
RUN_NAME   = (
    f"robertuito_base_{SEED}"
    if not USE_SLIDING
    else f"robertuito_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": MODEL_CKPT,
        "run_type": "finetuned_single_run",
        "max_len_train": MAX_LEN_TRAIN,
        "use_sliding": USE_SLIDING,
        "max_len_sw": MAX_LEN_SW,
        "stride_sw": STRIDE_SW,
        "agg_method": AGG_METHOD,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 4,
        "weight_decay": 0.01,
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "sliding" if USE_SLIDING else "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(
    text,
    model,
    tokenizer,
    id2label,
    max_len=128,
    stride=64,
    agg="mean"
):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"robertuito_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Usar siempre el mejor checkpoint cargado por Trainer
best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("\nExcel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

# =========================================
# 14) Resumen final legible
# =========================================
print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: robertuito_slide_1405_mean ID: unfdv60k USE_SLIDING: True AGG: mean
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3245', 'grad_norm': '16.33', 'learning_rate': '3.952e-06', 'epoch': '3.226'}
{'eval_loss': '0.5971', 'eval_accuracy': '0.8115', 'eval_f1_macro': '0.8086', 'eval_f1_weighted': '0.811', 'eval_runtime': '0.2461', 'eval_samples_per_second': '495.7', 'eval_steps_per_second': '16.25', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '100.2', 'train_samples_per_second': '39.08', 'train_steps_per_second': '2.475', 'train_loss': '0.4549', 'epoch': '4'}

VALIDATION SIMPLE
  Accuracy: 0.8443
  F1-macro: 0.8431
              precision    recall  f1-score   support

           N       0.86      0.78      0.82        41
         Neu       0.87      0.87      0.87        45
           P       0.80      0.89      0.84        36

    accuracy                           0.84       122
   macro avg       0.84      0.85      0.84       122
weighted avg       0.85      0.84      0.84       122


VALIDATION SLIDING
  Accuracy: 0.8197
  F1-macro: 0.8158
              precision    recall  f1-score   support

           N       0.75      0.88      0.81        41
         Neu       0.90      0.84      0.87        45
           P       0.81      0.72      0.76        36

    accuracy                           0.82       122
   macro avg       0.82      0.81      0.82       122
weighted avg       0.83      0.82      0.

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▃█▁▁
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


# Con SEED 2050


## Truncado

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = False
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 2050
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
RUN_NAME   = (
    f"robertuito_base_{SEED}"
    if not USE_SLIDING
    else f"robertuito_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": MODEL_CKPT,
        "run_type": "finetuned_single_run",
        "max_len_train": MAX_LEN_TRAIN,
        "use_sliding": USE_SLIDING,
        "max_len_sw": MAX_LEN_SW,
        "stride_sw": STRIDE_SW,
        "agg_method": AGG_METHOD,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 4,
        "weight_decay": 0.01,
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "sliding" if USE_SLIDING else "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(
    text,
    model,
    tokenizer,
    id2label,
    max_len=128,
    stride=64,
    agg="mean"
):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"robertuito_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Usar siempre el mejor checkpoint cargado por Trainer
best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("\nExcel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

# =========================================
# 14) Resumen final legible
# =========================================
print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: robertuito_base_2050 ID: sqgal75p USE_SLIDING: False AGG: mean
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3509', 'grad_norm': '8.333', 'learning_rate': '3.952e-06', 'epoch': '3.226'}
{'eval_loss': '0.846', 'eval_accuracy': '0.7049', 'eval_f1_macro': '0.6995', 'eval_f1_weighted': '0.7035', 'eval_runtime': '0.2401', 'eval_samples_per_second': '508.1', 'eval_steps_per_second': '16.66', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '84.41', 'train_samples_per_second': '46.39', 'train_steps_per_second': '2.938', 'train_loss': '0.4583', 'epoch': '4'}

VALIDATION SIMPLE
  Accuracy: 0.7131
  F1-macro: 0.7076
              precision    recall  f1-score   support

           N       0.74      0.76      0.75        41
         Neu       0.74      0.76      0.75        45
           P       0.65      0.61      0.63        36

    accuracy                           0.71       122
   macro avg       0.71      0.71      0.71       122
weighted avg       0.71      0.71      0.71       122


TEST SIMPLE
  Accuracy: 0.7886
  F1-macro: 0.7854
              precision    recall  f1-score   support

           N       0.74      0.78      0.76        41
         Neu       0.86      0.83      0.84        46
           P       0.75      0.75      0.75        36

    accuracy                           0.79       123
   macro avg       0.79      0.79      0.79       123
weighted avg       0.79      0.79      0.79     

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁▆█▆
eval/accuracy_simple,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_f1_simple,▁
eval/class_N_precision_simple,▁
eval/class_N_recall_simple,▁
eval/class_N_support_simple,▁
+48,...


## Slide + Vote

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "vote"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 2050
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
RUN_NAME   = (
    f"robertuito_base_{SEED}"
    if not USE_SLIDING
    else f"robertuito_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": MODEL_CKPT,
        "run_type": "finetuned_single_run",
        "max_len_train": MAX_LEN_TRAIN,
        "use_sliding": USE_SLIDING,
        "max_len_sw": MAX_LEN_SW,
        "stride_sw": STRIDE_SW,
        "agg_method": AGG_METHOD,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 4,
        "weight_decay": 0.01,
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "sliding" if USE_SLIDING else "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(
    text,
    model,
    tokenizer,
    id2label,
    max_len=128,
    stride=64,
    agg="mean"
):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"robertuito_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Usar siempre el mejor checkpoint cargado por Trainer
best_model = trainer.model

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("\nExcel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

# =========================================
# 14) Resumen final legible
# =========================================
print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: robertuito_slide_2050_vote ID: 2izzg7s4 USE_SLIDING: True AGG: vote
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3509', 'grad_norm': '8.333', 'learning_rate': '3.952e-06', 'epoch': '3.226'}
{'eval_loss': '0.846', 'eval_accuracy': '0.7049', 'eval_f1_macro': '0.6995', 'eval_f1_weighted': '0.7035', 'eval_runtime': '0.2346', 'eval_samples_per_second': '520', 'eval_steps_per_second': '17.05', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '76.26', 'train_samples_per_second': '51.35', 'train_steps_per_second': '3.252', 'train_loss': '0.4583', 'epoch': '4'}

VALIDATION SIMPLE
  Accuracy: 0.7131
  F1-macro: 0.7076
              precision    recall  f1-score   support

           N       0.74      0.76      0.75        41
         Neu       0.74      0.76      0.75        45
           P       0.65      0.61      0.63        36

    accuracy                           0.71       122
   macro avg       0.71      0.71      0.71       122
weighted avg       0.71      0.71      0.71       122


VALIDATION SLIDING
  Accuracy: 0.7213
  F1-macro: 0.7150
              precision    recall  f1-score   support

           N       0.69      0.83      0.76        41
         Neu       0.73      0.73      0.73        45
           P       0.75      0.58      0.66        36

    accuracy                           0.72       122
   macro avg       0.73      0.72      0.72       122
weighted avg       0.72      0.72      0.

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁▆█▆
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


## Slide + Mean

In [ ]:
# =========================================
# 0) W&B + Config (antes de todo)
# =========================================
import os
import random
import numpy as np
import pandas as pd
import torch
import wandb

WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY

if wandb.run is not None:
    wandb.finish()

USE_SLIDING = True
AGG_METHOD  = "mean"   # "mean" / "vote"

# =========================================
# 1) Imports + Config
# =========================================
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

SEED       = 2050
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
RUN_NAME   = (
    f"robertuito_base_{SEED}"
    if not USE_SLIDING
    else f"robertuito_slide_{SEED}_{AGG_METHOD}"
)

MAX_LEN_TRAIN = 128
MAX_LEN_SW    = 128
STRIDE_SW     = 96

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================================
# 2) Init W&B
# =========================================
run = wandb.init(
    project=WANDB_PROJECT,
    # entity=WANDB_ENTITY,
    name=RUN_NAME,
    config={
        "seed": SEED,
        "model_ckpt": MODEL_CKPT,
        "run_type": "finetuned_single_run",
        "max_len_train": MAX_LEN_TRAIN,
        "use_sliding": USE_SLIDING,
        "max_len_sw": MAX_LEN_SW,
        "stride_sw": STRIDE_SW,
        "agg_method": AGG_METHOD,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 4,
        "weight_decay": 0.01,
        "selection_metric": "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple",
    },
    tags=["finetune", "single_run", "sliding" if USE_SLIDING else "simple"],
)

print("RUN:", wandb.run.name, "ID:", wandb.run.id, "USE_SLIDING:", USE_SLIDING, "AGG:", AGG_METHOD)

# =========================================
# 3) Utils
# =========================================
def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

@torch.no_grad()
def predict_proba_windows(
    text: str,
    model,
    tokenizer,
    max_len: int = 128,
    stride: int = 64,
    device=None
):
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "max":
        return probs_windows.max(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean', 'max' o 'vote'")

def predict_text_with_sliding(
    text,
    model,
    tokenizer,
    id2label,
    max_len=128,
    stride=64,
    agg="mean"
):
    """
    Retorna:
      pred_label (str),
      conf (float),
      probs_agg (np.array),
      n_win (int),
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )
    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)
        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

# =========================================
# 4) Datos
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

# Split 80/10/10
train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

wandb.log({
    "data/n_train": len(train_df),
    "data/n_val": len(val_df),
    "data/n_test": len(test_df),
})

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================
# 5) Tokenize
# =========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

def tokenize_train(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN_TRAIN)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

ds = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})
ds_tok = ds.map(tokenize_train, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================================
# 6) Model
# =========================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# =========================================
# 7) Metrics para Trainer
# =========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 8) Trainer + Train
# =========================================
use_fp16 = torch.cuda.is_available()
OUTPUT_DIR = f"robertuito_finetuned_{wandb.run.id}"

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    fp16=use_fp16,
    report_to="wandb",
    run_name=wandb.run.name,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Usar siempre el mejor checkpoint cargado por Trainer
best_model = trainer.model

# =========================================
# Guardar modelo en Google Drive
# =========================================
SAVE_MODEL_DIR = Path("/content/drive/MyDrive/Tesis/robertuito_finetuned_parlamento_seed2050")
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(SAVE_MODEL_DIR))
tokenizer.save_pretrained(str(SAVE_MODEL_DIR))
print(f"✅ Modelo guardado en: {SAVE_MODEL_DIR}")

# =========================================
# 9) VALIDATION simple — referencia, no selección final
# =========================================
pred_val_simple = trainer.predict(ds_tok["validation"])
logits_val_simple = pred_val_simple.predictions
y_true_val_simple = pred_val_simple.label_ids
y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

eval_simple_metrics, eval_simple_report = build_metrics_dict(
    y_true=y_true_val_simple,
    y_pred=y_pred_val_simple,
    label_list=label_list,
    label2id=label2id,
    split="eval",
    variant="simple"
)

wandb.log(eval_simple_metrics)

print("\nVALIDATION SIMPLE")
print(f"  Accuracy: {eval_simple_metrics['eval/accuracy_simple']:.4f}")
print(f"  F1-macro: {eval_simple_metrics['eval/f1_macro_simple']:.4f}")
print(classification_report(
    y_true_val_simple,
    y_pred_val_simple,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 10) VALIDATION sliding — referencia, no selección final
# =========================================
if USE_SLIDING:
    y_true_val_sliding = val_df["label"].to_numpy()
    y_pred_val_sw = []
    nwin_val = []

    for txt in val_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_val_sw.append(label2id[lab])
        nwin_val.append(n_win)

    y_pred_val_sw = np.array(y_pred_val_sw)

    eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
        y_true=y_true_val_sliding,
        y_pred=y_pred_val_sw,
        label_list=label_list,
        label2id=label2id,
        split="eval",
        variant="sliding"
    )

    eval_sliding_metrics.update({
        "eval/avg_windows_sliding": float(np.mean(nwin_val)),
        "eval/max_windows_sliding": float(np.max(nwin_val)),
    })

    wandb.log(eval_sliding_metrics)

    print("\nVALIDATION SLIDING")
    print(f"  Accuracy: {eval_sliding_metrics['eval/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {eval_sliding_metrics['eval/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true_val_sliding,
        y_pred_val_sw,
        target_names=label_list,
        zero_division=0
    ))

# =========================================
# 11) TEST simple — resultado final comparable
# =========================================
pred_test_simple = trainer.predict(ds_tok["test"])
logits_test_simple = pred_test_simple.predictions
y_true = pred_test_simple.label_ids
y_pred = np.argmax(logits_test_simple, axis=-1)
probs = softmax_stable(logits_test_simple, axis=1)

true_label_txt = np.array([id2label[i] for i in y_true])
pred_label_txt = np.array([id2label[i] for i in y_pred])

test_meta = test_df.reset_index(drop=True).copy()
test_meta["true_label"] = true_label_txt
test_meta["pred_label"] = pred_label_txt
test_meta["conf_pred"] = probs.max(axis=1)
test_meta["proba_N"] = probs[:, label2id["N"]]
test_meta["proba_Neu"] = probs[:, label2id["Neu"]]
test_meta["proba_P"] = probs[:, label2id["P"]]
test_meta["proba_vector"] = probs.tolist()

test_simple_metrics, test_simple_report = build_metrics_dict(
    y_true=y_true,
    y_pred=y_pred,
    label_list=label_list,
    label2id=label2id,
    split="test",
    variant="simple"
)

wandb.log(test_simple_metrics)

print("\nTEST SIMPLE")
print(f"  Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f}")
print(f"  F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f}")
print(classification_report(
    y_true,
    y_pred,
    target_names=label_list,
    zero_division=0
))

# =========================================
# 12) TEST sliding — resultado final comparable
# =========================================
if USE_SLIDING:
    y_pred_sw = []
    conf_sw = []
    nwin_sw = []
    votes_N_list, votes_Neu_list, votes_P_list = [], [], []

    for txt in test_df["text"].tolist():
        lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
            txt,
            best_model,
            tokenizer,
            id2label,
            max_len=MAX_LEN_SW,
            stride=STRIDE_SW,
            agg=AGG_METHOD
        )
        y_pred_sw.append(label2id[lab])
        conf_sw.append(conf)
        nwin_sw.append(n_win)

        if votes_counts is None:
            votes_N_list.append(np.nan)
            votes_Neu_list.append(np.nan)
            votes_P_list.append(np.nan)
        else:
            votes_N_list.append(int(votes_counts[label2id["N"]]))
            votes_Neu_list.append(int(votes_counts[label2id["Neu"]]))
            votes_P_list.append(int(votes_counts[label2id["P"]]))

    y_pred_sw = np.array(y_pred_sw)

    assert len(test_df) == len(y_true) == len(y_pred_sw), "Mismatch en longitudes de test"

    test_meta["pred_label_sw"] = [id2label[i] for i in y_pred_sw]
    test_meta["conf_sw"] = conf_sw
    test_meta["n_windows"] = nwin_sw
    test_meta["votes_N"] = votes_N_list
    test_meta["votes_Neu"] = votes_Neu_list
    test_meta["votes_P"] = votes_P_list

    test_sliding_metrics, test_sliding_report = build_metrics_dict(
        y_true=y_true,
        y_pred=y_pred_sw,
        label_list=label_list,
        label2id=label2id,
        split="test",
        variant="sliding"
    )

    test_sliding_metrics.update({
        "test/avg_windows_sliding": float(np.mean(nwin_sw)),
        "test/max_windows_sliding": float(np.max(nwin_sw)),
    })

    wandb.log(test_sliding_metrics)

    print("\nTEST SLIDING")
    print(f"  Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f}")
    print(f"  F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f}")
    print(classification_report(
        y_true,
        y_pred_sw,
        target_names=label_list,
        zero_division=0
    ))

else:
    test_meta["pred_label_sw"] = np.nan
    test_meta["conf_sw"] = np.nan
    test_meta["n_windows"] = np.nan
    test_meta["votes_N"] = np.nan
    test_meta["votes_Neu"] = np.nan
    test_meta["votes_P"] = np.nan

# =========================================
# 13) Export Excel
# =========================================
cols_excel = [
    "id_intervencion", "file_name", "n_legislatura", "cuerpo", "fecha",
    "locutor", "encabezado", "intervencion", "n_palabras", "n_caracteres",
    "año", "sexo_locutor", "apellido_locutor_norm", "partido_imputado",
    "sentimiento", "text",
    "true_label",
    "pred_label", "conf_pred", "proba_N", "proba_Neu", "proba_P", "proba_vector",
    "pred_label_sw", "conf_sw", "n_windows", "votes_N", "votes_Neu", "votes_P",
]
cols_excel = [c for c in cols_excel if c in test_meta.columns]

df_errors_simple = test_meta[test_meta["true_label"] != test_meta["pred_label"]].copy()
df_errors_sliding = (
    test_meta[test_meta["true_label"] != test_meta["pred_label_sw"]].copy()
    if USE_SLIDING else pd.DataFrame()
)
df_corrige_sliding = (
    test_meta[
        (test_meta["true_label"] != test_meta["pred_label"]) &
        (test_meta["true_label"] == test_meta["pred_label_sw"])
    ].copy()
    if USE_SLIDING else pd.DataFrame()
)

out_path = f"analisis_{RUN_NAME}.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    test_meta[cols_excel].to_excel(writer, index=False, sheet_name="test_all")
    df_errors_simple[cols_excel].to_excel(writer, index=False, sheet_name="errores_simple")
    if USE_SLIDING:
        df_errors_sliding[cols_excel].to_excel(writer, index=False, sheet_name="errores_sliding")
        df_corrige_sliding[cols_excel].to_excel(writer, index=False, sheet_name="corrige_sliding")

print("\nExcel generado:", out_path)
print(f"  test total:        {len(test_meta)}")
print(f"  errores simple:    {len(df_errors_simple)}")
if USE_SLIDING:
    print(f"  errores sliding:   {len(df_errors_sliding)}")
    print(f"  corrige sliding:   {len(df_corrige_sliding)}")

# =========================================
# 14) Resumen final legible
# =========================================
print("\n" + "=" * 60)
print("RESUMEN FINAL — TEST")
print(f"Simple  | Accuracy: {test_simple_metrics['test/accuracy_simple']:.4f} | "
      f"F1-macro: {test_simple_metrics['test/f1_macro_simple']:.4f} | "
      f"F1-weighted: {test_simple_metrics['test/f1_weighted_simple']:.4f}")

if USE_SLIDING:
    print(f"Sliding | Accuracy: {test_sliding_metrics['test/accuracy_sliding']:.4f} | "
          f"F1-macro: {test_sliding_metrics['test/f1_macro_sliding']:.4f} | "
          f"F1-weighted: {test_sliding_metrics['test/f1_weighted_sliding']:.4f}")
print("=" * 60)

wandb.finish()

RUN: robertuito_slide_2050_mean ID: gt9qcamk USE_SLIDING: True AGG: mean
Train: 979 Val: 122 Test: 123


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.3509', 'grad_norm': '8.333', 'learning_rate': '3.952e-06', 'epoch': '3.226'}
{'eval_loss': '0.846', 'eval_accuracy': '0.7049', 'eval_f1_macro': '0.6995', 'eval_f1_weighted': '0.7035', 'eval_runtime': '0.2355', 'eval_samples_per_second': '518', 'eval_steps_per_second': '16.98', 'epoch': '4'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '89.83', 'train_samples_per_second': '43.59', 'train_steps_per_second': '2.761', 'train_loss': '0.4583', 'epoch': '4'}

VALIDATION SIMPLE
  Accuracy: 0.7131
  F1-macro: 0.7076
              precision    recall  f1-score   support

           N       0.74      0.76      0.75        41
         Neu       0.74      0.76      0.75        45
           P       0.65      0.61      0.63        36

    accuracy                           0.71       122
   macro avg       0.71      0.71      0.71       122
weighted avg       0.71      0.71      0.71       122


VALIDATION SLIDING
  Accuracy: 0.7213
  F1-macro: 0.7150
              precision    recall  f1-score   support

           N       0.69      0.83      0.76        41
         Neu       0.73      0.73      0.73        45
           P       0.75      0.58      0.66        36

    accuracy                           0.72       122
   macro avg       0.73      0.72      0.72       122
weighted avg       0.72      0.72      0.

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,▁▆█▆
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...


# Random Search

In [7]:
# =========================================
# 0) DEBUG CUDA
# =========================================
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# =========================================
# 1) Imports + Config general
# =========================================
import gc
import random
import shutil
import traceback
import numpy as np
import pandas as pd
import wandb
import torch

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
)

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

# =========================================
# 2) Config global
# =========================================
WANDB_PROJECT = "Tesis ORT"
WANDB_ENTITY  = "fbenedetti97-universidad-ort-uruguay"  # opcional

os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["WANDB_ENTITY"] = WANDB_ENTITY  # descomentá si aplica

if wandb.run is not None:
    wandb.finish()

BASE_SEED  = 1899
MODEL_CKPT = "pysentimiento/robertuito-sentiment-analysis"
DATA_PATH  = Path("/content/df_final_para_modelo.jsonl")  # opcional / informativo
#AGG_METHOD  = "mean"

USE_SLIDING = True

# --- Random search ---
N_RANDOM_RUNS = 3
RUN_TEST_EACH_EXPERIMENT = True   # False: test solo con el mejor config al final

# --- Multi-seed final ---
FINAL_SEEDS = [1899, 1405, 2050]

DELETE_OUTPUT_DIR = False

# Seed cuyo modelo se guarda en Google Drive para inferencia
SAVE_BEST_SEED = 2050
SAVE_MODEL_DIR = Path("/content/drive/MyDrive/Tesis/robertuito_finetuned_parlamento_seed2050")

# Métrica oficial para elegir la mejor config en FASE 1
SELECTION_METRIC = "eval_f1_macro_sliding" if USE_SLIDING else "eval_f1_macro_simple"

# =========================================
# 3) Utils
# =========================================
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def softmax_stable(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def assert_labels_ok(df, df_name="df"):
    assert df["label"].notna().all(), f"{df_name}: hay labels NaN"
    uniq = set(df["label"].unique())
    assert uniq <= {0, 1, 2}, f"{df_name}: labels fuera de rango -> {uniq}"

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def normalize_config_types(cfg: dict):
    return {
        "learning_rate":    float(cfg["learning_rate"]),
        "epochs":           int(cfg["epochs"]),
        "train_batch_size": int(cfg["train_batch_size"]),
        "eval_batch_size":  int(cfg["eval_batch_size"]),
        "weight_decay":     float(cfg["weight_decay"]),
        "warmup_ratio":     float(cfg["warmup_ratio"]),
        "max_len_train":    int(cfg["max_len_train"]),
        "max_len_sw":       int(cfg["max_len_sw"]),
        "stride_sw":        int(cfg["stride_sw"]),
        "agg_method":       str(cfg["agg_method"]),
    }

@torch.no_grad()
def predict_proba_windows(text: str, model, tokenizer, max_len: int = 128, stride: int = 64, device=None):
    """
    Sliding windows usando overflow del tokenizer.
    stride = cantidad de tokens solapados entre ventanas.
    """
    model.eval()
    if device is None:
        device = model.device

    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_len,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True,
        padding=False,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]

    if len(input_ids) > 0 and isinstance(input_ids[0], int):
        input_ids = [input_ids]
        attention_mask = [attention_mask]

    max_batch_len = max(len(x) for x in input_ids)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 1

    def pad(seq, pad_value):
        return seq + [pad_value] * (max_batch_len - len(seq))

    input_ids = torch.tensor(
        [pad(x, pad_id) for x in input_ids],
        dtype=torch.long,
        device=device
    )
    attention_mask = torch.tensor(
        [pad(x, 0) for x in attention_mask],
        dtype=torch.long,
        device=device
    )

    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

    return probs, len(probs)

def aggregate_probs(probs_windows: np.ndarray, method: str = "mean"):
    if method == "mean":
        return probs_windows.mean(axis=0)
    if method == "vote":
        votes = np.argmax(probs_windows, axis=1)
        counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(float)
        return counts / counts.sum()
    raise ValueError("method debe ser 'mean' o 'vote'")

def predict_text_with_sliding(text, model, tokenizer, id2label, max_len=128, stride=64, agg="mean"):
    """
    Retorna:
      pred_label   (str)
      conf         (float)
      probs_agg    (np.array)
      n_win        (int)
      votes_counts (np.array int) o None
    """
    probs_windows, n_win = predict_proba_windows(
        text=text,
        model=model,
        tokenizer=tokenizer,
        max_len=max_len,
        stride=stride
    )

    votes_counts = None

    if agg == "vote":
        votes = np.argmax(probs_windows, axis=1)
        votes_counts = np.bincount(votes, minlength=probs_windows.shape[1]).astype(int)

        max_count = votes_counts.max()
        tied = np.where(votes_counts == max_count)[0]

        if len(tied) == 1:
            pred_id = int(tied[0])
        else:
            mean_probs = probs_windows.mean(axis=0)
            pred_id = int(tied[np.argmax(mean_probs[tied])])

        probs_agg = votes_counts.astype(float) / votes_counts.sum()
        conf = float(probs_agg[pred_id])
        pred_label = id2label[pred_id]
        return pred_label, conf, probs_agg, n_win, votes_counts

    probs_agg = aggregate_probs(probs_windows, method=agg)
    pred_id = int(np.argmax(probs_agg))
    pred_label = id2label[pred_id]
    conf = float(np.max(probs_agg))
    return pred_label, conf, probs_agg, n_win, votes_counts

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

# =========================================
# 4) Load data (una sola vez)
# =========================================
df_lab = df_limpio[df_limpio["sentimiento"].notna()].copy()
df_lab["text"] = df_lab["intervencion"].astype(str)

valid_labels = ["N", "Neu", "P"]
df_lab = df_lab[df_lab["sentimiento"].isin(valid_labels)].copy()

label_list = ["N", "Neu", "P"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
df_lab["label"] = df_lab["sentimiento"].map(label2id)

print("Labels únicas en df_lab:", sorted(df_lab["label"].unique().tolist()))
print("Conteo labels:")
print(df_lab["label"].value_counts(dropna=False).sort_index())

assert_labels_ok(df_lab, "df_lab")

train_df, temp_df = train_test_split(
    df_lab,
    test_size=0.20,
    random_state=BASE_SEED,
    stratify=df_lab["label"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=BASE_SEED,
    stratify=temp_df["label"]
)

assert_labels_ok(train_df, "train_df")
assert_labels_ok(val_df, "val_df")
assert_labels_ok(test_df, "test_df")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# =========================================
# 5) Espacio de búsqueda
# =========================================
SEARCH_SPACE = {
    "learning_rate":    [1e-5, 1.5e-5, 2e-5, 3e-5],
    "epochs":           [3, 4, 5],
    "train_batch_size": [8, 16],
    "eval_batch_size":  [32],
    "weight_decay":     [0.0, 0.01, 0.05],
    "warmup_ratio":     [0.0, 0.1],
    "max_len_train":    [128, 256],
    "max_len_sw":       [128, 256],
    "agg_method":       ["mean"],
}

STRIDE_BY_MAXLEN = {
    128: [32, 64, 96],
    256: [64, 128, 192],
}

def sample_unique_configs(n_runs: int, seed: int = 1899):
    rng = random.Random(seed)
    seen = set()
    configs = []

    while len(configs) < n_runs:
        max_len_sw = rng.choice(SEARCH_SPACE["max_len_sw"])

        cfg = {
            "learning_rate":    rng.choice(SEARCH_SPACE["learning_rate"]),
            "epochs":           rng.choice(SEARCH_SPACE["epochs"]),
            "train_batch_size": rng.choice(SEARCH_SPACE["train_batch_size"]),
            "eval_batch_size":  rng.choice(SEARCH_SPACE["eval_batch_size"]),
            "weight_decay":     rng.choice(SEARCH_SPACE["weight_decay"]),
            "warmup_ratio":     rng.choice(SEARCH_SPACE["warmup_ratio"]),
            "max_len_train":    rng.choice(SEARCH_SPACE["max_len_train"]),
            "max_len_sw":       max_len_sw,
            "stride_sw":        rng.choice(STRIDE_BY_MAXLEN[max_len_sw]),
            "agg_method":       rng.choice(SEARCH_SPACE["agg_method"]),
        }

        key = tuple(sorted(cfg.items()))
        if key not in seen:
            seen.add(key)
            configs.append(cfg)

    return configs

random_configs = sample_unique_configs(N_RANDOM_RUNS, seed=BASE_SEED)

print("\nConfiguraciones sorteadas:")
for i, cfg in enumerate(random_configs, 1):
    print(f"  {i}. {cfg}")

# =========================================
# 6) Función principal por experimento
# =========================================
def run_one_experiment(cfg: dict, exp_idx: int, seed: int, run_prefix: str = "rs"):
    """
    Entrena una sola corrida con la config y seed dadas.

    run_prefix:
      - "rs"    -> random search (fase 1)
      - "final" -> multi-seed final (fase 2)

    En fase 1 se usa VALIDATION para selección de configs.
    En fase 2 se corre TEST para evaluación final.
    """
    cfg = normalize_config_types(cfg)
    set_all_seeds(seed)

    run_name = (
        f"{run_prefix}_{exp_idx:02d}"
        f"_seed{seed}"
        f"_agg{cfg['agg_method']}"
        f"_lr{cfg['learning_rate']}"
        f"_mlt{cfg['max_len_train']}"
        f"_mls{cfg['max_len_sw']}"
        f"_st{cfg['stride_sw']}"
    )

    if run_prefix == "rs":
        run_tags = ["random_search", "validation_only", "sliding" if USE_SLIDING else "simple"]
        selection_metric_value = SELECTION_METRIC
    else:
        run_tags = ["final_multiseed", "test", "sliding" if USE_SLIDING else "simple"]
        selection_metric_value = "test/f1_macro_sliding" if USE_SLIDING else "test/f1_macro_simple"

    run = wandb.init(
        project=WANDB_PROJECT,
        # entity=WANDB_ENTITY,
        name=run_name,
        config={
            "seed": seed,
            "model_ckpt": MODEL_CKPT,
            "use_sliding": USE_SLIDING,
            "run_prefix": run_prefix,
            "selection_metric": selection_metric_value,
            **cfg,
        },
        tags=run_tags,
        reinit="finish_previous",
    )

    print("\n" + "=" * 90)
    print(f"RUN: {run.name} | ID: {run.id} | prefix: {run_prefix}")
    print(cfg)

    wandb.log({
        "data/n_train": len(train_df),
        "data/n_val": len(val_df),
        "data/n_test": len(test_df),
    })

    output_dir = f"robertuito_{run_prefix}_{run.id}"

    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)

        def tokenize_fn(batch):
            return tokenizer(batch["text"], truncation=True, max_length=cfg["max_len_train"])

        train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
        val_ds   = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
        test_ds  = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

        ds = DatasetDict({
            "train": train_ds,
            "validation": val_ds,
            "test": test_ds
        })
        ds_tok = ds.map(tokenize_fn, batched=True)

        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

        assert set(ds_tok["train"]["label"]) <= {0, 1, 2}
        assert set(ds_tok["validation"]["label"]) <= {0, 1, 2}
        assert set(ds_tok["test"]["label"]) <= {0, 1, 2}

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_CKPT,
            num_labels=len(label_list),
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )

        assert model.config.num_labels == 3

        use_fp16 = torch.cuda.is_available()

        args = TrainingArguments(
            output_dir=output_dir,
            learning_rate=cfg["learning_rate"],
            per_device_train_batch_size=cfg["train_batch_size"],
            per_device_eval_batch_size=cfg["eval_batch_size"],
            num_train_epochs=cfg["epochs"],
            weight_decay=cfg["weight_decay"],
            warmup_ratio=cfg["warmup_ratio"],
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            greater_is_better=True,
            seed=seed,
            logging_steps=50,
            fp16=use_fp16,
            report_to="wandb",
            run_name=run.name,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=ds_tok["train"],
            eval_dataset=ds_tok["validation"],
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        best_model = trainer.model

        # ─── Guardar modelo del seed elegido en Google Drive ──────────────
        if run_prefix == "final" and seed == SAVE_BEST_SEED:
            print(f"
💾 Guardando modelo (seed {seed}) en {SAVE_MODEL_DIR} ...")
            SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
            trainer.save_model(str(SAVE_MODEL_DIR))
            tokenizer.save_pretrained(str(SAVE_MODEL_DIR))
            print("✅ Modelo guardado en Drive.")


        # =========================================
        # Métricas auxiliares del trainer (informativas)
        # =========================================
        eval_logs = [x for x in trainer.state.log_history if "eval_f1_macro" in x]
        if eval_logs:
            best_eval_trainer = max(eval_logs, key=lambda x: x["eval_f1_macro"])
        else:
            best_eval_trainer = {}

        result_row = {
            "status": "ok",
            "run_name": run.name,
            "run_id": run.id,
            "seed": seed,
            "run_prefix": run_prefix,
            **cfg,
            "trainer_best_eval_loss": best_eval_trainer.get("eval_loss"),
            "trainer_best_eval_accuracy": best_eval_trainer.get("eval_accuracy"),
            "trainer_best_eval_f1_macro": best_eval_trainer.get("eval_f1_macro"),
            "trainer_best_eval_f1_weighted": best_eval_trainer.get("eval_f1_weighted"),
            "trainer_best_eval_epoch": best_eval_trainer.get("epoch"),
        }

        # =========================================
        # VALIDATION simple — oficial para comparación interna
        # =========================================
        pred_val_simple = trainer.predict(ds_tok["validation"])
        logits_val_simple = pred_val_simple.predictions
        y_true_val_simple = pred_val_simple.label_ids
        y_pred_val_simple = np.argmax(logits_val_simple, axis=-1)

        eval_simple_metrics, eval_simple_report = build_metrics_dict(
            y_true=y_true_val_simple,
            y_pred=y_pred_val_simple,
            label_list=label_list,
            label2id=label2id,
            split="eval",
            variant="simple"
        )

        wandb.log(eval_simple_metrics)

        result_row.update({
            "eval_accuracy_simple": eval_simple_metrics["eval/accuracy_simple"],
            "eval_f1_macro_simple": eval_simple_metrics["eval/f1_macro_simple"],
            "eval_f1_weighted_simple": eval_simple_metrics["eval/f1_weighted_simple"],
        })

        for lbl in label_list:
            result_row[f"eval_class_{lbl}_precision_simple"] = eval_simple_metrics[f"eval/class_{lbl}_precision_simple"]
            result_row[f"eval_class_{lbl}_recall_simple"] = eval_simple_metrics[f"eval/class_{lbl}_recall_simple"]
            result_row[f"eval_class_{lbl}_f1_simple"] = eval_simple_metrics[f"eval/class_{lbl}_f1_simple"]
            result_row[f"eval_class_{lbl}_support_simple"] = eval_simple_metrics[f"eval/class_{lbl}_support_simple"]
            result_row[f"eval_class_{lbl}_accuracy_ovr_simple"] = eval_simple_metrics[f"eval/class_{lbl}_accuracy_ovr_simple"]

        # =========================================
        # VALIDATION sliding — métrica principal si USE_SLIDING=True
        # =========================================
        if USE_SLIDING:
            y_true_val_sliding = val_df["label"].to_numpy()
            y_pred_val_sw_list = []
            nwin_val_list = []

            for txt in val_df["text"].tolist():
                lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
                    txt,
                    best_model,
                    tokenizer,
                    id2label,
                    max_len=cfg["max_len_sw"],
                    stride=cfg["stride_sw"],
                    agg=cfg["agg_method"],
                )
                y_pred_val_sw_list.append(label2id[lab])
                nwin_val_list.append(n_win)

            y_pred_val_sw = np.array(y_pred_val_sw_list)

            eval_sliding_metrics, eval_sliding_report = build_metrics_dict(
                y_true=y_true_val_sliding,
                y_pred=y_pred_val_sw,
                label_list=label_list,
                label2id=label2id,
                split="eval",
                variant="sliding"
            )

            eval_sliding_metrics.update({
                "eval/avg_windows_sliding": float(np.mean(nwin_val_list)),
                "eval/max_windows_sliding": float(np.max(nwin_val_list)),
            })

            wandb.log(eval_sliding_metrics)

            result_row.update({
                "eval_accuracy_sliding": eval_sliding_metrics["eval/accuracy_sliding"],
                "eval_f1_macro_sliding": eval_sliding_metrics["eval/f1_macro_sliding"],
                "eval_f1_weighted_sliding": eval_sliding_metrics["eval/f1_weighted_sliding"],
                "eval_avg_windows_sliding": eval_sliding_metrics["eval/avg_windows_sliding"],
                "eval_max_windows_sliding": eval_sliding_metrics["eval/max_windows_sliding"],
            })

            for lbl in label_list:
                result_row[f"eval_class_{lbl}_precision_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_precision_sliding"]
                result_row[f"eval_class_{lbl}_recall_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_recall_sliding"]
                result_row[f"eval_class_{lbl}_f1_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_f1_sliding"]
                result_row[f"eval_class_{lbl}_support_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_support_sliding"]
                result_row[f"eval_class_{lbl}_accuracy_ovr_sliding"] = eval_sliding_metrics[f"eval/class_{lbl}_accuracy_ovr_sliding"]

        # =========================================
        # TEST solo en fase final (o si se fuerza explícitamente)
        # =========================================
        run_test = (run_prefix == "final") or RUN_TEST_EACH_EXPERIMENT

        if run_test:
            # -----------------------------------------
            # TEST simple
            # -----------------------------------------
            pred_test_simple = trainer.predict(ds_tok["test"])
            logits_test_simple = pred_test_simple.predictions
            y_true = pred_test_simple.label_ids
            y_pred = np.argmax(logits_test_simple, axis=-1)
            probs_test_simple = softmax_stable(logits_test_simple, axis=1)

            test_simple_metrics, test_simple_report = build_metrics_dict(
                y_true=y_true,
                y_pred=y_pred,
                label_list=label_list,
                label2id=label2id,
                split="test",
                variant="simple"
            )

            wandb.log(test_simple_metrics)

            result_row.update({
                "test_accuracy_simple": test_simple_metrics["test/accuracy_simple"],
                "test_f1_macro_simple": test_simple_metrics["test/f1_macro_simple"],
                "test_f1_weighted_simple": test_simple_metrics["test/f1_weighted_simple"],
            })

            for lbl in label_list:
                result_row[f"test_class_{lbl}_precision_simple"] = test_simple_metrics[f"test/class_{lbl}_precision_simple"]
                result_row[f"test_class_{lbl}_recall_simple"] = test_simple_metrics[f"test/class_{lbl}_recall_simple"]
                result_row[f"test_class_{lbl}_f1_simple"] = test_simple_metrics[f"test/class_{lbl}_f1_simple"]
                result_row[f"test_class_{lbl}_support_simple"] = test_simple_metrics[f"test/class_{lbl}_support_simple"]
                result_row[f"test_class_{lbl}_accuracy_ovr_simple"] = test_simple_metrics[f"test/class_{lbl}_accuracy_ovr_simple"]

            # -----------------------------------------
            # TEST sliding
            # -----------------------------------------
            if USE_SLIDING:
                y_pred_sw_list = []
                conf_sw_list = []
                nwin_sw_list = []

                for txt in test_df["text"].tolist():
                    lab, conf, probs_agg, n_win, votes_counts = predict_text_with_sliding(
                        txt,
                        best_model,
                        tokenizer,
                        id2label,
                        max_len=cfg["max_len_sw"],
                        stride=cfg["stride_sw"],
                        agg=cfg["agg_method"],
                    )
                    y_pred_sw_list.append(label2id[lab])
                    conf_sw_list.append(conf)
                    nwin_sw_list.append(n_win)

                y_pred_sw = np.array(y_pred_sw_list)

                test_sliding_metrics, test_sliding_report = build_metrics_dict(
                    y_true=y_true,
                    y_pred=y_pred_sw,
                    label_list=label_list,
                    label2id=label2id,
                    split="test",
                    variant="sliding"
                )

                test_sliding_metrics.update({
                    "test/avg_windows_sliding": float(np.mean(nwin_sw_list)),
                    "test/max_windows_sliding": float(np.max(nwin_sw_list)),
                })

                wandb.log(test_sliding_metrics)

                result_row.update({
                    "test_accuracy_sliding": test_sliding_metrics["test/accuracy_sliding"],
                    "test_f1_macro_sliding": test_sliding_metrics["test/f1_macro_sliding"],
                    "test_f1_weighted_sliding": test_sliding_metrics["test/f1_weighted_sliding"],
                    "test_avg_windows_sliding": test_sliding_metrics["test/avg_windows_sliding"],
                    "test_max_windows_sliding": test_sliding_metrics["test/max_windows_sliding"],
                })

                for lbl in label_list:
                    result_row[f"test_class_{lbl}_precision_sliding"] = test_sliding_metrics[f"test/class_{lbl}_precision_sliding"]
                    result_row[f"test_class_{lbl}_recall_sliding"] = test_sliding_metrics[f"test/class_{lbl}_recall_sliding"]
                    result_row[f"test_class_{lbl}_f1_sliding"] = test_sliding_metrics[f"test/class_{lbl}_f1_sliding"]
                    result_row[f"test_class_{lbl}_support_sliding"] = test_sliding_metrics[f"test/class_{lbl}_support_sliding"]
                    result_row[f"test_class_{lbl}_accuracy_ovr_sliding"] = test_sliding_metrics[f"test/class_{lbl}_accuracy_ovr_sliding"]

        wandb.finish()

        del trainer, model, best_model, ds_tok, ds, train_ds, val_ds, test_ds
        cleanup_cuda()

        if DELETE_OUTPUT_DIR and os.path.exists(output_dir):
            shutil.rmtree(output_dir, ignore_errors=True)

        return result_row

    except Exception as e:
        err_trace = traceback.format_exc()
        print("\n[ERROR EN RUN]")
        print(type(e).__name__, str(e))
        print(err_trace)

        try:
            wandb.log({
                "error": 1,
                "error_type": type(e).__name__,
                "error_msg": str(e),
            })
            wandb.finish()
        except:
            pass

        cleanup_cuda()

        if DELETE_OUTPUT_DIR and os.path.exists(output_dir):
            shutil.rmtree(output_dir, ignore_errors=True)

        return {
            "status": "error",
            "run_name": run.name if wandb.run is not None else run_name,
            "run_id": run.id if "run" in locals() else None,
            "seed": seed,
            "run_prefix": run_prefix,
            **cfg,
            "error_type": type(e).__name__,
            "error_msg": str(e),
        }

# =========================================
# 7) Random search
# =========================================
print("\n" + "=" * 90)
print(f"FASE 1 — RANDOM SEARCH ({N_RANDOM_RUNS} runs, seed fija={BASE_SEED})")
print(f"Métrica de selección: {SELECTION_METRIC}")
print("=" * 90)

rs_results = []

for i, cfg in enumerate(random_configs, start=1):
    row = run_one_experiment(cfg, exp_idx=i, seed=BASE_SEED, run_prefix="rs")
    rs_results.append(row)

rs_df = pd.DataFrame(rs_results)

sort_col = SELECTION_METRIC

if sort_col in rs_df.columns:
    rs_df = rs_df.sort_values(sort_col, ascending=False, na_position="last").reset_index(drop=True)

rs_df.to_csv("robertuito_random_search_resultados.csv", index=False)
print("\nCSV guardado: robertuito_random_search_resultados.csv")

cols_show = [
    "run_name",
    sort_col,
    "eval_f1_macro_simple",
    "eval_f1_macro_sliding",
    "learning_rate",
    "epochs",
    "train_batch_size",
    "weight_decay",
    "warmup_ratio",
    "max_len_train",
    "max_len_sw",
    "stride_sw",
    "agg_method",
]
print(rs_df[[c for c in cols_show if c in rs_df.columns]])

# =========================================
# 8) Seleccionar mejor config
# =========================================
rs_ok = rs_df[rs_df["status"] == "ok"].copy()

if len(rs_ok) == 0:
    raise RuntimeError("No hubo corridas exitosas en el random search.")

best_row = rs_ok.sort_values(sort_col, ascending=False).iloc[0]

cfg_keys = list(SEARCH_SPACE.keys()) + ["stride_sw"]
best_cfg_raw = {k: best_row[k] for k in cfg_keys if k in best_row}
best_cfg = normalize_config_types(best_cfg_raw)

print("\n" + "=" * 90)
print("MEJOR CONFIGURACIÓN (por validation):")
for k, v in best_cfg.items():
    print(f"  {k}: {v}")
print(f"  {sort_col}: {best_row[sort_col]:.4f}")

print("\nTipos de best_cfg:")
for k, v in best_cfg.items():
    print(f"  {k}: {type(v)}")

# =========================================
# 9) Multi-seed final con la mejor config
# =========================================
print("\n" + "=" * 90)
print(f"FASE 2 — MULTI-SEED FINAL ({len(FINAL_SEEDS)} seeds: {FINAL_SEEDS})")
print("Misma config, distinta seed -> estimación de varianza del fine-tuning")
print("Estas corridas SÍ corren test set.")
print("=" * 90)

final_results = []

for i, seed in enumerate(FINAL_SEEDS, start=1):
    row = run_one_experiment(best_cfg, exp_idx=i, seed=seed, run_prefix="final")
    final_results.append(row)

final_df = pd.DataFrame(final_results)
final_df.to_csv("robertuito_final_multiseed_resultados.csv", index=False)
print("\nCSV guardado: robertuito_final_multiseed_resultados.csv")

# =========================================
# 10) Resumen estadístico final
# =========================================
final_ok = final_df[final_df["status"] == "ok"].copy()

print("\n" + "=" * 90)
print("RESUMEN MULTI-SEED (TEST SET)")

report_cols = [
    "test_f1_macro_simple",
    "test_f1_weighted_simple",
    "test_accuracy_simple",
    "test_f1_macro_sliding",
    "test_f1_weighted_sliding",
    "test_accuracy_sliding",
]

for col in report_cols:
    if col in final_ok.columns and final_ok[col].notna().any():
        vals = final_ok[col].dropna().values
        print(
            f"  {col}: mean={np.mean(vals):.4f}  std={np.std(vals):.4f}  "
            f"min={np.min(vals):.4f}  max={np.max(vals):.4f}"
        )

print("\nDone.")

data/n_test,▁
data/n_train,▁
data/n_val,▁
data/n_test,123
data/n_train,979
data/n_val,122


Labels únicas en df_lab: [0, 1, 2]
Conteo labels:
label
0    410
1    452
2    362
Name: count, dtype: int64
Train: 979 | Val: 122 | Test: 123

Configuraciones sorteadas:
  1. {'learning_rate': 1e-05, 'epochs': 3, 'train_batch_size': 16, 'eval_batch_size': 32, 'weight_decay': 0.0, 'warmup_ratio': 0.0, 'max_len_train': 128, 'max_len_sw': 128, 'stride_sw': 64, 'agg_method': 'mean'}
  2. {'learning_rate': 1e-05, 'epochs': 5, 'train_batch_size': 16, 'eval_batch_size': 32, 'weight_decay': 0.0, 'warmup_ratio': 0.1, 'max_len_train': 128, 'max_len_sw': 128, 'stride_sw': 96, 'agg_method': 'mean'}
  3. {'learning_rate': 1e-05, 'epochs': 3, 'train_batch_size': 8, 'eval_batch_size': 32, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'max_len_train': 128, 'max_len_sw': 128, 'stride_sw': 64, 'agg_method': 'mean'}

FASE 1 — RANDOM SEARCH (3 runs, seed fija=1899)
Métrica de selección: eval_f1_macro_sliding



RUN: rs_01_seed1899_aggmean_lr1e-05_mlt128_mls128_st64 | ID: em5nrt7y | prefix: rs
{'learning_rate': 1e-05, 'epochs': 3, 'train_batch_size': 16, 'eval_batch_size': 32, 'weight_decay': 0.0, 'warmup_ratio': 0.0, 'max_len_train': 128, 'max_len_sw': 128, 'stride_sw': 64, 'agg_method': 'mean'}


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/122 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

data/n_test,▁
data/n_train,▁
data/n_val,▁
eval/accuracy,██▁
eval/accuracy_simple,▁
eval/accuracy_sliding,▁
eval/avg_windows_sliding,▁
eval/class_N_accuracy_ovr_simple,▁
eval/class_N_accuracy_ovr_sliding,▁
eval/class_N_f1_simple,▁
+88,...



CSV guardado: robertuito_final_multiseed_resultados.csv

RESUMEN MULTI-SEED (TEST SET)
  test_f1_macro_simple: mean=0.7162  std=0.0085  min=0.7094  max=0.7282
  test_f1_weighted_simple: mean=0.7196  std=0.0084  min=0.7134  max=0.7315
  test_accuracy_simple: mean=0.7209  std=0.0077  min=0.7154  max=0.7317
  test_f1_macro_sliding: mean=0.7279  std=0.0101  min=0.7149  max=0.7395
  test_f1_weighted_sliding: mean=0.7327  std=0.0101  min=0.7195  max=0.7439
  test_accuracy_sliding: mean=0.7371  std=0.0101  min=0.7236  max=0.7480

Done.
